# **Project: Senior citizen Identification**

**Problem Statement:**  In this task, you will develop a machine learning model to predict multiple persons in a video or real-time webcam feed for a mall or local store. Your model should detect their age and gender. If a person’s age is more than 60, mark them as a senior citizen and detect their gender. After detection, store the age, gender, and time of visit in an Excel or CSV file. Guidelines: Create your own machine learning model for this task. While a graphical user interface (GUI) is not mandatory, you are welcome to include one if you wish. Although accuracy is important, we will evaluate your work based on the overall performance of your model and the successful functionality of your GUI.

# Downloading Data and Fetchig 5000 images

In [ ]:
import os
import kagglehub
import shutil
import random
import pandas as pd

# 1. Download/Locate the dataset
print("Downloading/Locating dataset...")
path = kagglehub.dataset_download("jangedoo/utkface-new")

# 2. Define your target destination
target_data_dir = '/content/master_data'

# 3. Clear existing folder and create fresh directory
if os.path.exists(target_data_dir):
    shutil.rmtree(target_data_dir)
    print("🧹 Cleared existing master_data folder.")
os.makedirs(target_data_dir)

# 4. Prepare Source Path
source_path = os.path.join(path, 'UTKFace')
if not os.path.exists(source_path):
    source_path = path

# 5. Select 5,000 random images and copy
all_files = [f for f in os.listdir(source_path) if f.endswith('.jpg')]
sample_size = 5000

if len(all_files) >= sample_size:
    selected_files = random.sample(all_files, sample_size)

    print(f"✅ Copying exactly {sample_size} images to {target_data_dir}...")
    for file in selected_files:
        shutil.copy(os.path.join(source_path, file), os.path.join(target_data_dir, file))
    print("✅ Successfully copied 5,000 images.")


In [ ]:
final_count = len([f for f in os.listdir(target_data_dir) if f.endswith('.jpg')])
print(f"Final image count in {target_data_dir}: {final_count}")

# Creating Labelled CSVs

In [ ]:
import pandas as pd
import os

def generate_labels(root_dir):
    data = []

    # List all files in the master_data folder
    files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]

    for filename in files:
        try:
            # UTKFace format: age_gender_race_date.jpg
            parts = filename.split('_')
            age = int(parts[0])
            gender = int(parts[1])  # 0: Male, 1: Female

            data.append({
                'filename': filename,
                'age': age,
                'gender': gender
            })
        except (ValueError, IndexError):
            # Skip files that don't follow the naming convention
            continue

    # Save to CSV
    df = pd.DataFrame(data)
    df.to_csv('master_data.csv', index=False)
    print(f"✅ CSV generated with {len(df)} entries.")
    return df

# Execute
df = generate_labels('/content/master_data')
print(df.head())

# Splitting Data

In [ ]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Setup paths
source_dir = '/content/master_data'
base_output_dir = '/content/master_data_split'
splits = ['train', 'test', 'val']

# 2. Create directory structure
for split in splits:
    os.makedirs(os.path.join(base_output_dir, split), exist_ok=True)

# 3. Load your master metadata
df = pd.read_csv('master_data.csv')

# 4. Perform the splits
train_df, rem_df = train_test_split(df, test_size=0.3, random_state=42)
test_df, val_df = train_test_split(rem_df, test_size=0.5, random_state=42)

# 5. Copy files and save CSVs
split_dfs = {'train': train_df, 'test': test_df, 'val': val_df}

for split_name, split_df in split_dfs.items():
    print(f"Processing {split_name} set ({len(split_df)} images)...")

    # Create a copy to avoid SettingWithCopy warnings
    new_df = split_df.copy()

    # Copy files
    for idx, row in new_df.iterrows():
        # FIX: Join source_dir with the filename
        src = os.path.join(source_dir, row['filename'])
        dst = os.path.join(base_output_dir, split_name, row['filename'])
        shutil.copy(src, dst)

    # Update path in dataframe
    new_df['filename'] = os.path.join(base_output_dir, split_name) + '/' + new_df['filename']

    # Save CSV
    csv_name = f'master_data_{split_name}.csv'
    new_df.to_csv(csv_name, index=False)
    print(f"✅ Created {csv_name}")

print("Dataset distribution complete!")

# Doenloading Dataset

In [ ]:
import shutil

# Define the folder to zip and the name for the output file
folder_to_zip = '/content/master_data_split'
output_filename = '/content/master_data_split'

# Create the zip archive
# format can be 'zip', 'tar', 'gztar', 'bztar', or 'xztar'
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"✅ Successfully zipped {folder_to_zip} into {output_filename}.zip")

# Custom Model

# Loading Data

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os

# 1. Define transformation pipeline and batch size
# These are essential for converting images into tensors for your model
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Adjust size to match your model input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

batch_size = 32

class AgeGenderDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        # Assuming gender is encoded as 'Male': 0, 'Female': 1
        # Adjust these keys if your CSV uses different naming
        self.gender_map = {'Male': 0, 'Female': 1, 'M': 0, 'F': 1}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load image (Ensure row['filename'] or path column is correct)
        image = Image.open(row['filename']).convert('RGB')

        # Gender (Categorical)
        gender_label = self.gender_map.get(str(row['gender']), 0)

        # Age (Continuous/Regression)
        age_label = float(row['age'])

        if self.transform:
            image = self.transform(image)

        # Return image and the two targets
        # Age is float, Gender is long (for classification)
        return image, torch.tensor(age_label, dtype=torch.float32), torch.tensor(gender_label, dtype=torch.long)



# 1. Re-create your loaders
train_dataset = AgeGenderDataset('master_data_train.csv', transform=transform)
val_dataset = AgeGenderDataset('master_data_val.csv', transform=transform)
test_dataset = AgeGenderDataset('master_data_test.csv', transform=transform)

# Note: In a real scenario, split your CSV into train/val/test first
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ train_test_val_loader updated successfully to support Age and Gender.")

# Building Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AgeGenderModel(nn.Module):
    def __init__(self):
        super(AgeGenderModel, self).__init__()

        # --- Shared Feature Extraction ---
        # Input: 64x64x3
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # After 3 pooling layers of 64x64: 64 -> 32 -> 16 -> 8
        # Flatten size: 128 channels * 8 * 8 = 8192
        self.flatten_size = 128 * 8 * 8

        # --- Head 1: Gender Classification (Binary) ---
        self.gender_fc1 = nn.Linear(self.flatten_size, 128)
        self.gender_fc2 = nn.Linear(128, 1) # Output for Sigmoid (0 or 1)

        # --- Head 2: Age Regression (Continuous) ---
        self.age_fc1 = nn.Linear(self.flatten_size, 128)
        self.age_fc2 = nn.Linear(128, 1) # Single output for Age

    def forward(self, x):
        # Shared Convolutional Base
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten
        x = x.view(x.size(0), -1)

        # Branch 1: Gender (Use sigmoid at the end for binary classification)
        gender_out = F.relu(self.gender_fc1(x))
        gender_out = torch.sigmoid(self.gender_fc2(gender_out))

        # Branch 2: Age
        age_out = F.relu(self.age_fc1(x))
        age_out = self.age_fc2(age_out)

        return age_out, gender_out

# Initialize
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeGenderModel().to(device)

# Verify the architecture
print(f"✅ Multi-task model initialized on {device}")

# Training Model

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn

# 1. Initialize history storage
history = {'gender_loss': [], 'age_loss': [], 'total_loss': []}

# Loss functions and Optimizer
# BCELoss is for binary classification (Gender)
# MSELoss is for regression (Age)
gender_criterion = nn.BCELoss()
age_criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_multitask(num_epochs=10):
    print("Starting training...")
    for epoch in range(num_epochs):
        model.train()
        running_total_loss = 0.0
        running_gender_loss = 0.0
        running_age_loss = 0.0

        for images, age_labels, gender_labels in train_loader:
            # Move to device
            images = images.to(device)
            age_labels = age_labels.to(device).float()
            gender_labels = gender_labels.to(device).float().unsqueeze(1) # Ensure shape [batch, 1]

            optimizer.zero_grad()

            # Forward pass
            # Note: Model returns (age_out, gender_out)
            age_pred, gender_pred = model(images)

            # Calculate individual losses
            loss_gender = gender_criterion(gender_pred, gender_labels)
            loss_age = age_criterion(age_pred, age_labels)

            # Combined Loss (Weight age loss if necessary, e.g., 0.1 * loss_age)
            loss = loss_gender + (0.1 * loss_age)

            loss.backward()
            optimizer.step()

            # Accumulate metrics
            running_total_loss += loss.item()
            running_gender_loss += loss_gender.item()
            running_age_loss += loss_age.item()

        # Calculate averages for the epoch
        avg_total = running_total_loss / len(train_loader)
        avg_gender = running_gender_loss / len(train_loader)
        avg_age = running_age_loss / len(train_loader)

        # Save to history
        history['total_loss'].append(avg_total)
        history['gender_loss'].append(avg_gender)
        history['age_loss'].append(avg_age)

        print(f"Epoch {epoch+1}/{num_epochs} | Total Loss: {avg_total:.4f} | "
              f"Gender Loss: {avg_gender:.4f} | Age Loss: {avg_age:.4f}")

    # Save the model
    torch.save(model.state_dict(), 'age_gender_model.pth')
    print("✅ Training complete. Model saved as 'age_gender_model.pth'")

# Run the training
train_multitask(num_epochs=10)

# Visualization

In [ ]:
import matplotlib.pyplot as plt

def plot_all_histories(history):
    # Create a figure with a grid layout
    fig = plt.figure(figsize=(14, 10))

    # 1. Dual-Axis Plot (Top Row, spanning 2 columns)
    # Using Gender Loss and Age Loss
    ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Gender Loss (BCE)', color='tab:blue')
    ax1.plot(history['gender_loss'], color='tab:blue', label='Gender Loss', marker='o')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Age Loss (MSE)', color='tab:red')
    ax2.plot(history['age_loss'], color='tab:red', label='Age Loss', marker='s')
    ax2.tick_params(axis='y', labelcolor='tab:red')
    ax2.set_title('Multi-Task Training Loss over Epochs (Dual-Axis)')

    # 2. Gender Loss (Bottom Left)
    ax3 = plt.subplot2grid((2, 2), (1, 0))
    ax3.plot(history['gender_loss'], color='tab:blue', marker='o', label='Gender Loss')
    ax3.set_title('Gender Classification Loss')
    ax3.set_xlabel('Epochs'); ax3.set_ylabel('Binary CrossEntropy')
    ax3.grid(True, linestyle='--', alpha=0.6)

    # 3. Age Loss (Bottom Right)
    ax4 = plt.subplot2grid((2, 2), (1, 1))
    ax4.plot(history['age_loss'], color='tab:red', marker='s', label='Age Loss')
    ax4.set_title('Age Regression Loss')
    ax4.set_xlabel('Epochs'); ax4.set_ylabel('MSE')
    ax4.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

# Call this after your training loop
plot_all_histories(history)

# Testing & Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import torch

def evaluate_model(model, test_loader, device):
    model.eval()
    all_gender_preds = []
    all_gender_targets = []
    all_age_preds = []
    all_age_targets = []

    with torch.no_grad():
        for images, age_labels, gender_labels in test_loader:
            images = images.to(device)
            # Model returns (age_out, gender_out)
            age_pred, gender_pred = model(images)

            # Gender Predictions
            predicted_gender = (gender_pred > 0.5).float()
            all_gender_preds.extend(predicted_gender.cpu().numpy().flatten())
            all_gender_targets.extend(gender_labels.numpy())

            # Age Predictions - FIX: Ensure we always have an iterable
            # .view(-1) forces a 1D array even if the batch size is 1
            preds = age_pred.squeeze().cpu().numpy()
            if preds.ndim == 0:
                all_age_preds.append(preds.item())
            else:
                all_age_preds.extend(preds)

            all_age_targets.extend(age_labels.numpy())

    # 1. Confusion Matrix for Gender
    cm = confusion_matrix(all_gender_targets, all_gender_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=['Male', 'Female'],
                yticklabels=['Male', 'Female'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Gender Confusion Matrix')
    plt.show()

    # 2. Accuracy Visualization
    acc = accuracy_score(all_gender_targets, all_gender_preds)
    print(f"✅ Gender Classification Accuracy: {acc*100:.2f}%")

    # 3. Age Prediction Visualization
    plt.figure(figsize=(8, 5))
    plt.scatter(all_age_targets, all_age_preds, alpha=0.5, color='purple')
    plt.plot([min(all_age_targets), max(all_age_targets)],
             [min(all_age_targets), max(all_age_targets)],
             color='red', linestyle='--')
    plt.xlabel('Actual Age')
    plt.ylabel('Predicted Age')
    plt.title('Age Regression: Predicted vs Actual')
    plt.show()

# Execute evaluation
evaluate_model(model, test_loader, device)

# **Github Repo**




https://github.com/KVAlwaysLearning/Senior_citizen_Identification_Sub

# **Streamlit App**


https://seniorcitizenidentificationsub-app.streamlit.app/